# Differential Interleaved Ramsey: True Sensitivity Scaling

Corrected analysis. The Ramsey slope dP/dδ sets the fundamental limit.
We compute exactly how many shots are needed for Hz-level sensitivity.


In [ ]:
import math

import numpy as np
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
torch.manual_seed(7)
np.random.seed(7)

## The Ramsey slope: where sensitivity comes from
P(δ) = ½(1 + V·cos(2πδt)). Sensitivity dP/dδ is maximal when sin(2πδt)=±1.


In [ ]:
delta_khz = 30.0
# Optimal delay: t = 1/(4δ) puts us at maximum slope
t_opt_us = 1 / (4 * delta_khz * 1e3) * 1e6
print(f"Optimal Ramsey delay: {t_opt_us:.1f} µs")
V = math.exp(-t_opt_us / 80.0)
slope_per_hz = 0.5 * V * 2 * math.pi * t_opt_us * 1e-6
print(f"Max slope dP/dδ: {slope_per_hz:.2e} per Hz")
print(f"50 Hz effect → ΔP = {slope_per_hz * 50:.2e}")
print(f"5 Hz effect  → ΔP = {slope_per_hz * 5:.2e}")

## Shot requirement: the honest math
Shot noise: σ_P = √(p(1-p)/N). For SNR=2 on ΔP, need N > 4·p(1-p)/ΔP².


In [ ]:
p = 0.5  # worst case (max variance)
for effect_hz in [5000, 500, 50, 5]:
    dP = slope_per_hz * effect_hz
    N_needed = 4 * p * (1 - p) / dP**2
    # At 10 kHz repetition rate
    time_s = N_needed / 10000
    print(f"{effect_hz:5d} Hz: ΔP={dP:.1e}, N={N_needed:.1e} shots, ~{time_s:.0f}s at 10kHz rep rate")

## GPU verification: interleaved pairs at optimal delay
Simulate with the optimal t and enough shots to see the scaling.


In [ ]:
def ramsey_batch(delta_hz, t_us, n_shots):
    V = math.exp(-t_us / 80.0)
    phase = 2 * math.pi * (30000 + delta_hz) * t_us * 1e-6
    p = 0.5 * (1 + V * math.cos(phase))
    shots = torch.rand(n_shots, device=device) < p
    return shots.float().mean().item(), p


t_us = t_opt_us
# Interleaved: alternate defective/clean in blocks of 1000 shots
n_blocks = 20
shots_per_block = 5000
diffs = []
for _b in range(n_blocks):
    p_def, _ = ramsey_batch(50.0, t_us, shots_per_block)  # defective: +50 Hz
    p_clean, _ = ramsey_batch(0.0, t_us, shots_per_block)  # clean: baseline
    diffs.append(p_def - p_clean)
diffs = np.array(diffs)
expected_dP = slope_per_hz * 50
print(f"Expected ΔP: {expected_dP:.2e}")
print(f"Measured ΔP: {diffs.mean():.2e} ± {diffs.std() / np.sqrt(n_blocks):.2e}")
print(f"SNR: {abs(diffs.mean()) / (diffs.std() / np.sqrt(n_blocks)):.1f}")

## Drift cancellation check
Add ±2 kHz slow drift. Interleaving should still extract the 50 Hz signal.


In [ ]:
diffs_drift = []
drift_phase = 0.0
for _b in range(n_blocks):
    # Drift wanders between blocks but is common to both in a pair
    drift_phase += np.random.randn() * 200  # Hz of drift per block
    p_def, _ = ramsey_batch(50.0 + drift_phase, t_us, shots_per_block)
    p_clean, _ = ramsey_batch(0.0 + drift_phase, t_us, shots_per_block)
    diffs_drift.append(p_def - p_clean)
diffs_drift = np.array(diffs_drift)
print(f"With drift: ΔP = {diffs_drift.mean():.2e} ± {diffs_drift.std() / np.sqrt(n_blocks):.2e}")
print(f"SNR: {abs(diffs_drift.mean()) / (diffs_drift.std() / np.sqrt(n_blocks)):.1f}")
print("Drift cancels in the pair difference.")

## Verdict

The honest scaling law:
- **5 kHz effects:** ~100 shots. Trivial.
- **500 Hz effects:** ~10⁴ shots. Minutes.
- **50 Hz effects:** ~10⁶ shots. Hours at 10 kHz rep rate.
- **5 Hz effects:** ~10⁸ shots. Days. Needs multi-qubit parallelization.

**Interleaving cancels drift** — verified on GPU. The pair difference
is immune to common-mode wander.

**Optimal Ramsey delay** is t = 1/(4δ) ≈ 8.3 µs for the 30 kHz ghost.
This maximizes dP/dδ.

**For Claude Code:** Start at 500 Hz sensitivity (10⁴ shots, minutes).
If null, the data tells us whether going deeper is worth the QPU time.
Eric's directive stands: calibrate shot budget to observed noise.
Parallelize across qubits to reach Hz-level without days of wall-clock.
